# Deep Learning Lab — Fashion-MNIST with a Multilayer Perceptron

## From 28×28 pixels to a 10-class prediction

In this lab, a multilayer perceptron (MLP) will classify grayscale clothing images from **Fashion-MNIST**.

The complete flow is:

**image → normalize → flatten → hidden layers → class probabilities → predicted class**

The main objective is not only to obtain an accuracy value. The code should make every stage of the neural network understandable:

1. what enters the network;
2. what each Dense layer computes;
3. why nonlinear activation is required;
4. why the output layer has 10 neurons;
5. how Softmax converts outputs into probabilities;
6. how the loss guides training;
7. how training and validation curves are interpreted;
8. how predictions and errors are inspected.

## Learning outcomes

By the end of the lab, you should be able to:

- describe the Fashion-MNIST input and output structure;
- explain why a $28\times28$ image becomes a vector of 784 values for an MLP;
- build a Keras MLP with configurable hidden layers;
- explain the role of Dense, ReLU, Dropout, Softmax, Adam, and cross-entropy;
- train the network using training and validation data;
- interpret accuracy and loss curves;
- evaluate the final model on unseen test data;
- inspect class probabilities, correct predictions, and mistakes;
- compare two MLP architectures and justify which one generalizes better.

## Quick glossary

| Term | Meaning |
|---|---|
| **Sample** | One image and its label. |
| **Feature** | One numeric input value. In the flattened image, each pixel becomes one feature. |
| **Label** | Correct class index from 0 to 9. |
| **Batch** | A small group of samples processed before one parameter update. |
| **Epoch** | One complete pass through the training set. |
| **Dense layer** | Every neuron receives input from every output of the previous layer. |
| **Weight** | Trainable value that controls the influence of an input. |
| **Bias** | Trainable offset added before activation. |
| **Activation** | Nonlinear transformation applied after the weighted sum. |
| **ReLU** | $\operatorname{ReLU}(z)=\max(0,z)$. |
| **Logit** | Raw output value before Softmax. |
| **Softmax** | Converts the final 10 logits into probabilities that sum to 1. |
| **Loss** | Numeric training objective that the optimizer attempts to reduce. |
| **Optimizer** | Algorithm that updates the trainable weights and biases. |
| **Learning rate** | Size of an optimizer update step. |
| **Dropout** | During training, randomly disables a fraction of activations to reduce over-reliance on particular neurons. |
| **Training set** | Data used to update model parameters. |
| **Validation set** | Data used to monitor generalization during training. |
| **Test set** | Unseen data used for final evaluation. |
| **Underfitting** | Model is too limited to learn the useful pattern. |
| **Overfitting** | Model fits training data much better than unseen data. |

## Lab rule

This notebook is intentionally incomplete.

The missing lines are the important neural-network steps.  
Each task includes enough information to complete it immediately.

Work from top to bottom.  
When a stage is correct, the next stage becomes meaningful.

In [ ]:
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)

In [ ]:
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

NUM_CLASSES = len(CLASS_NAMES)
IMAGE_HEIGHT = 28
IMAGE_WIDTH = 28
FLAT_FEATURES = IMAGE_HEIGHT * IMAGE_WIDTH

print("Classes:", NUM_CLASSES)
print("Pixels per image:", FLAT_FEATURES)

In [ ]:
# Keras provides Fashion-MNIST directly.
# The first call may download the dataset if it is not already cached.
(train_images_raw, train_labels_raw), (test_images_raw, test_labels) = (
    keras.datasets.fashion_mnist.load_data()
)

print("Raw training images:", train_images_raw.shape)
print("Raw training labels:", train_labels_raw.shape)
print("Raw test images:", test_images_raw.shape)
print("Raw test labels:", test_labels.shape)
print("Pixel-value range:", train_images_raw.min(), "to", train_images_raw.max())

# Stage 1 — Understand the data

Fashion-MNIST contains grayscale $28\times28$ images.

Each label is one integer from 0 to 9.

Before building a network, inspect several raw samples and connect the integer label to the class name.

In [ ]:
def show_image_grid(images, labels, class_names, count=12):
    count = min(count, len(images))

    cols = 4
    rows = int(np.ceil(count / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.5 * rows))
    axes = np.asarray(axes).reshape(-1)

    for i in range(count):
        axes[i].imshow(images[i], cmap="gray")
        axes[i].set_title(f"{labels[i]} · {class_names[int(labels[i])]}")
        axes[i].axis("off")

    for i in range(count, len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


show_image_grid(
    train_images_raw,
    train_labels_raw,
    CLASS_NAMES,
    count=12,
)

## Concept check

For one Fashion-MNIST image:

- height = 28 pixels;
- width = 28 pixels;
- total pixel values =

$$
28\times28=784
$$

The raw pixel values are integers from 0 to 255.

The MLP will use normalized floating-point values in the range $[0,1]$.

# Stage 2 — Normalize and flatten

A Dense network expects a vector per sample.

Therefore:

$$
(28,28)
\longrightarrow
(784,)
$$

Normalization is performed using

$$
x_{\text{normalized}}
=
\frac{x}{255}
$$

### Your task

Complete the two operations:

1. divide the pixel values by `255.0`;
2. reshape each image into 784 values.

### Strong hints

```python
train_images_raw.astype("float32") / 255.0
```

and

```python
.reshape(-1, FLAT_FEATURES)
```

In [ ]:
# TODO: preprocess the training and test images.

train_images = None
test_images = None

# Suggested implementation:
#
# train_images = (
#     train_images_raw.astype("float32") / 255.0
# ).reshape(-1, FLAT_FEATURES)
#
# test_images = (
#     test_images_raw.astype("float32") / 255.0
# ).reshape(-1, FLAT_FEATURES)

if train_images is None or test_images is None:
    print("Complete normalization and flattening first.")
else:
    print("Training tensor:", train_images.shape)
    print("Test tensor:", test_images.shape)
    print("Normalized range:", train_images.min(), "to", train_images.max())

In [ ]:
def make_fast_lab_split(
    images,
    labels,
    train_size=12000,
    val_size=2000,
    seed=2026,
):
    """
    Shuffle the original training data and create a smaller split
    so the lab trains quickly on a CPU.
    """
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(images))

    train_idx = indices[:train_size]
    val_idx = indices[train_size:train_size + val_size]

    return (
        images[train_idx],
        labels[train_idx],
        images[val_idx],
        labels[val_idx],
    )

# Stage 3 — Create training and validation data

The original training set is shuffled first.

A smaller subset is used so experiments finish quickly:

- 12,000 training samples;
- 2,000 validation samples.

The test set remains separate and is not used for model fitting.

In [ ]:
if train_images is None:
    print("Complete Stage 2 first.")
else:
    train_x, train_y, val_x, val_y = make_fast_lab_split(
        train_images,
        train_labels_raw,
        train_size=12000,
        val_size=2000,
        seed=SEED,
    )

    print("train_x:", train_x.shape)
    print("train_y:", train_y.shape)
    print("val_x:", val_x.shape)
    print("val_y:", val_y.shape)

## What one Dense layer computes

For layer $\ell$,

$$
\mathbf{z}^{(\ell)}
=
\mathbf{W}^{(\ell)}
\mathbf{a}^{(\ell-1)}
+
\mathbf{b}^{(\ell)}
$$

Then an activation function is applied:

$$
\mathbf{a}^{(\ell)}
=
g\left(\mathbf{z}^{(\ell)}\right)
$$

### Symbols

- $\mathbf{a}^{(\ell-1)}$: input vector to the current layer;
- $\mathbf{W}^{(\ell)}$: trainable weight matrix;
- $\mathbf{b}^{(\ell)}$: trainable bias vector;
- $\mathbf{z}^{(\ell)}$: weighted sum before activation;
- $g(\cdot)$: activation function;
- $\mathbf{a}^{(\ell)}$: output of the layer.

For the hidden layers, ReLU is used:

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

ReLU allows the network to build nonlinear decision boundaries. Without nonlinear activations, stacking Dense layers would still reduce to one linear transformation.

## Why the output layer has 10 neurons

Fashion-MNIST contains 10 mutually exclusive classes.

The output layer therefore contains one neuron per class:

$$
[z_0,z_1,\ldots,z_9]
$$

Softmax converts those logits into probabilities:

$$
p_k
=
\frac{e^{z_k}}
{\displaystyle\sum_{j=0}^{9}e^{z_j}}
$$

with

$$
\sum_{k=0}^{9}p_k=1
$$

The predicted class is

$$
\hat{y}
=
\operatorname*{arg\,max}_{k} p_k
$$

Example:

$$
[0.01,\ 0.02,\ 0.03,\ 0.04,\ 0.05,\ 0.08,\ 0.10,\ 0.12,\ 0.50,\ 0.05]
$$

The largest probability is at index 8, so the predicted class is class 8, **Bag**.

In [ ]:
def draw_mlp_diagram(hidden_layers=(256, 128), num_classes=10):
    """
    Draw a simple block-level view of the MLP.
    This deliberately avoids drawing hundreds of individual neurons.
    """
    labels = [
        "Input\n784 pixels",
        *[f"Hidden\n{units} ReLU" for units in hidden_layers],
        f"Output\n{num_classes} Softmax",
    ]

    x_positions = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(11, 3.8))

    for i, (x, label) in enumerate(zip(x_positions, labels)):
        ax.scatter([x], [0], s=4200, edgecolor="black", linewidth=1.5, zorder=3)
        ax.text(x, 0, label, ha="center", va="center", fontsize=11, weight="bold")

        if i < len(labels) - 1:
            ax.annotate(
                "",
                xy=(x_positions[i + 1] - 0.22, 0),
                xytext=(x + 0.22, 0),
                arrowprops=dict(arrowstyle="->", lw=2),
            )

    ax.set_xlim(-0.7, len(labels) - 0.3)
    ax.set_ylim(-1.0, 1.0)
    ax.set_title("Multilayer Perceptron used in this lab")
    ax.axis("off")
    plt.show()


draw_mlp_diagram((256, 128), NUM_CLASSES)

## How many parameters are being trained?

A Dense layer with

- $n_{\text{in}}$ input values, and
- $n_{\text{out}}$ neurons

contains

$$
(n_{\text{in}}+1)n_{\text{out}}
$$

trainable parameters.

The extra $1$ represents one bias per output neuron.

For the baseline network:

### Input 784 → hidden 256

$$
(784+1)\times256
=
200{,}960
$$

### Hidden 256 → hidden 128

$$
(256+1)\times128
=
32{,}896
$$

### Hidden 128 → output 10

$$
(128+1)\times10
=
1{,}290
$$

Total:

$$
200{,}960+32{,}896+1{,}290
=
235{,}146
$$

Dropout has no trainable parameters.

# Stage 4 — Build the MLP

The baseline architecture is:

```text
784 inputs
   ↓
Dense(256, ReLU)
   ↓
Dropout(0.20)
   ↓
Dense(128, ReLU)
   ↓
Dropout(0.20)
   ↓
Dense(10, Softmax)
```

### Your task

Complete `build_mlp()`.

### Hints

Start with:

```python
model = keras.Sequential(name="fashion_mnist_mlp")
model.add(layers.Input(shape=(FLAT_FEATURES,)))
```

For every hidden-layer size:

```python
model.add(layers.Dense(units, activation=activation))
```

If dropout is greater than zero:

```python
model.add(layers.Dropout(dropout_rate))
```

The final classification layer is:

```python
model.add(layers.Dense(NUM_CLASSES, activation="softmax"))
```

Compile using:

```python
keras.optimizers.Adam(learning_rate=learning_rate)
```

with:

```python
loss="sparse_categorical_crossentropy"
metrics=["accuracy"]
```

In [ ]:
def build_mlp(
    hidden_layers=(256, 128),
    activation="relu",
    dropout_rate=0.20,
    learning_rate=1e-3,
):
    # TODO: create the Sequential model.
    model = None

    # ----------------------------------------------------------
    # Suggested structure:
    #
    # model = keras.Sequential(name="fashion_mnist_mlp")
    # model.add(layers.Input(shape=(FLAT_FEATURES,)))
    #
    # for units in hidden_layers:
    #     model.add(layers.Dense(units, activation=activation))
    #
    #     if dropout_rate > 0:
    #         model.add(layers.Dropout(dropout_rate))
    #
    # model.add(layers.Dense(NUM_CLASSES, activation="softmax"))
    #
    # model.compile(
    #     optimizer=keras.optimizers.Adam(
    #         learning_rate=learning_rate
    #     ),
    #     loss="sparse_categorical_crossentropy",
    #     metrics=["accuracy"],
    # )
    # ----------------------------------------------------------

    return model


model = build_mlp()

if model is None:
    print("Complete build_mlp() using the hints above.")
else:
    model.summary()

## Why `sparse_categorical_crossentropy` is used

The labels are stored as integer class indices such as

```text
9
0
3
...
```

rather than one-hot vectors.

For one sample with correct class $y$, the loss is

$$
\mathcal{L}
=
-\log(p_y)
$$

For $N$ samples,

$$
\mathcal{L}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log\left(p_{i,y_i}\right)
$$

A confident correct prediction gives a small loss.  
A confident incorrect prediction gives a large loss.

`SparseCategoricalCrossentropy` is therefore appropriate when the target is one integer class index and the model outputs a probability distribution over multiple classes.

# Stage 5 — Train the model

Training data updates the weights.

Validation data does **not** update the weights. It measures how well the current model generalizes during training.

### Your task

Use:

```python
model.fit(...)
```

with:

- `train_x`, `train_y`;
- `validation_data=(val_x, val_y)`;
- 8 epochs;
- batch size 128.

In [ ]:
EPOCHS = 8
BATCH_SIZE = 128

history = None

if model is None:
    print("Complete Stage 4 first.")
elif "train_x" not in globals():
    print("Complete the data stages first.")
else:
    # TODO:
    #
    # history = model.fit(
    #     train_x,
    #     train_y,
    #     validation_data=(val_x, val_y),
    #     epochs=EPOCHS,
    #     batch_size=BATCH_SIZE,
    #     verbose=1,
    # )
    #
    print("Complete the model.fit(...) call.")

In [ ]:
def plot_training_history(history):
    epochs = range(1, len(history.history["accuracy"]) + 1)

    plt.figure(figsize=(8, 4.5))
    plt.plot(epochs, history.history["accuracy"], marker="o", label="training")
    plt.plot(epochs, history.history["val_accuracy"], marker="o", label="validation")
    plt.title("Accuracy across epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 4.5))
    plt.plot(epochs, history.history["loss"], marker="o", label="training")
    plt.plot(epochs, history.history["val_loss"], marker="o", label="validation")
    plt.title("Loss across epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

In [ ]:
if history is None:
    print("Train the model first.")
else:
    plot_training_history(history)

## How to read the curves

### Healthy learning

- training accuracy rises;
- validation accuracy also rises;
- training and validation loss generally decrease.

### Underfitting

Both training and validation performance remain weak.

The model may need more capacity, a better representation, more training, or different optimization settings.

### Overfitting

Training performance keeps improving while validation performance stops improving or becomes worse.

A large gap between training and validation behavior is a warning sign.

# Stage 6 — Evaluate on unseen test data

The test set is used only after training.

### Your task

Run:

```python
model.evaluate(test_images, test_labels, verbose=0)
```

and interpret both returned values.

In [ ]:
test_loss = None
test_accuracy = None

if model is None or history is None:
    print("Train the model first.")
else:
    # TODO:
    #
    # test_loss, test_accuracy = model.evaluate(
    #     test_images,
    #     test_labels,
    #     verbose=0,
    # )
    #
    # print(f"Test loss: {test_loss:.4f}")
    # print(f"Test accuracy: {test_accuracy:.4f}")
    #
    print("Complete the model.evaluate(...) call.")

# Stage 7 — Convert probabilities into predictions

The model returns 10 probabilities for every image:

$$
[p_0,p_1,\ldots,p_9]
$$

The predicted class is the index of the largest probability.

### Your task

Complete:

```python
test_probabilities = model.predict(...)
predicted_labels = np.argmax(..., axis=1)
```

In [ ]:
test_probabilities = None
predicted_labels = None

if model is None or history is None:
    print("Train the model first.")
else:
    # TODO:
    #
    # test_probabilities = model.predict(
    #     test_images,
    #     verbose=0,
    # )
    #
    # predicted_labels = np.argmax(
    #     test_probabilities,
    #     axis=1,
    # )
    #
    print("Complete probability prediction and argmax.")

In [ ]:
def show_predictions(images_2d, true_labels, probabilities, class_names, count=12):
    predicted_labels = np.argmax(probabilities, axis=1)
    confidences = np.max(probabilities, axis=1)

    count = min(count, len(images_2d))
    cols = 4
    rows = int(np.ceil(count / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(11, 2.8 * rows))
    axes = np.asarray(axes).reshape(-1)

    for i in range(count):
        true_id = int(true_labels[i])
        pred_id = int(predicted_labels[i])

        axes[i].imshow(images_2d[i], cmap="gray")
        axes[i].set_title(
            f"True: {class_names[true_id]}\n"
            f"Pred: {class_names[pred_id]} ({confidences[i]:.2f})"
        )
        axes[i].axis("off")

    for i in range(count, len(axes)):
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()


def show_misclassified_examples(
    images_2d,
    true_labels,
    probabilities,
    class_names,
    count=12,
):
    predicted_labels = np.argmax(probabilities, axis=1)
    wrong_indices = np.where(predicted_labels != true_labels)[0]

    if len(wrong_indices) == 0:
        print("No misclassified samples found.")
        return

    selected = wrong_indices[:count]

    show_predictions(
        images_2d[selected],
        true_labels[selected],
        probabilities[selected],
        class_names,
        count=len(selected),
    )

In [ ]:
if test_probabilities is None:
    print("Complete Stage 7 first.")
else:
    show_predictions(
        test_images_raw,
        test_labels,
        test_probabilities,
        CLASS_NAMES,
        count=12,
    )

# Stage 8 — Confusion matrix and errors

Overall accuracy does not tell us which classes are confused.

A confusion matrix answers:

> Which true class is being predicted as which class?

In [ ]:
def plot_confusion_matrix(true_labels, predicted_labels, class_names):
    cm = tf.math.confusion_matrix(
        true_labels,
        predicted_labels,
        num_classes=len(class_names),
    ).numpy()

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(cm)

    ax.set_title("Confusion matrix on the test set")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)

    threshold = cm.max() / 2

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j] > 0:
                ax.text(
                    j,
                    i,
                    cm[i, j],
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    plt.colorbar(im, ax=ax, label="Number of samples")
    plt.tight_layout()
    plt.show()

    return cm

In [ ]:
if predicted_labels is None:
    print("Complete Stage 7 first.")
else:
    cm = plot_confusion_matrix(
        test_labels,
        predicted_labels,
        CLASS_NAMES,
    )

In [ ]:
if test_probabilities is None:
    print("Complete Stage 7 first.")
else:
    show_misclassified_examples(
        test_images_raw,
        test_labels,
        test_probabilities,
        CLASS_NAMES,
        count=12,
    )

# Stage 9 — Required architecture experiment

The baseline uses:

```python
hidden_layers=(256, 128)
dropout_rate=0.20
```

Build **one second model** using:

```python
hidden_layers=(64,)
dropout_rate=0.00
```

Train it for the same number of epochs.

Then compare:

1. final training accuracy;
2. final validation accuracy;
3. validation loss;
4. test accuracy;
5. training time.

### Question

Does the smaller model underfit relative to the baseline?

Do not answer from parameter count alone. Use the curves and test result.

In [ ]:
# TODO: required experiment.
#
# small_model = build_mlp(
#     hidden_layers=(64,),
#     dropout_rate=0.0,
# )
#
# start = time.perf_counter()
#
# small_history = small_model.fit(
#     train_x,
#     train_y,
#     validation_data=(val_x, val_y),
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     verbose=1,
# )
#
# small_time = time.perf_counter() - start
#
# small_test_loss, small_test_accuracy = small_model.evaluate(
#     test_images,
#     test_labels,
#     verbose=0,
# )
#
# print("Small model test accuracy:", small_test_accuracy)
# print("Training time:", small_time)

print("Complete the required architecture experiment.")

# Stage 10 — Activation functions

The baseline uses ReLU in the hidden layers.

The following visualization compares several common activations.

Focus on the shape of each function rather than memorizing every formula.

In [ ]:
z = np.linspace(-5, 5, 400)

activation_values = {
    "ReLU": tf.nn.relu(z).numpy(),
    "Tanh": tf.nn.tanh(z).numpy(),
    "Sigmoid": tf.nn.sigmoid(z).numpy(),
    "ELU": tf.nn.elu(z).numpy(),
}

plt.figure(figsize=(8, 5))
for name, values in activation_values.items():
    plt.plot(z, values, label=name)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.title("Common activation functions")
plt.xlabel("Input z")
plt.ylabel("Activation output")
plt.legend()
plt.show()

## Required interpretation

Answer inside your submitted notebook:

1. Why must a Fashion-MNIST image be flattened before entering this MLP?
2. Why does the final layer contain exactly 10 neurons?
3. What does Softmax provide that a normal hidden-layer activation does not?
4. Why is `sparse_categorical_crossentropy` appropriate for integer labels?
5. What would indicate underfitting in the training curves?
6. What would indicate overfitting?
7. Which classes appear most difficult for the model according to the confusion matrix?
8. Compare the baseline network with the smaller network using evidence.
9. State one limitation of using an MLP on image data.

### Final lab message

**An MLP turns pixels into class probabilities by repeatedly applying learned linear transformations and nonlinear activations.**